<a href="https://colab.research.google.com/github/mugalan/introduction-to-statistical-learning/blob/main/Gaussian_Process_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Overview

Gaussian Process Regression (GPR) is a Bayesian approach to regression where we treat the underlying function as a random variable. Unlike traditional regression, which searches for a single "best" set of parameters (like weights in a neural network), GPR maintains a distribution over all possible functions that could fit the data.

The problem can be broken down into three conceptual layers: **The Latent Process**, **The Noisy Observation**, and **The Optimal Inference**.

---

**The Latent Process (The "Signal"):**

We assume there exists an underlying, "clean" process $X_g$ (where $g$ could be time, space, or any input dimension). We define this process as a **Gaussian Process**:

* For any set of points, the values of $X_g$ follow a Multivariate Gaussian distribution.
* This process is entirely defined by its **Mean Function** $\mu_g$ (the average behavior) and its **Kernel** $\kappa(g, g')$ (which defines the "shape" or smoothness of the functions).

---

**The Noisy Observation (The "Measurement"):**

In the real world, we rarely see the clean signal $X_g$ directly. Instead, we observe $Y_g$, which is the signal corrupted by random noise $\nu_g$.

* **The Problem:** We have a collection of $n$ noisy data points $\mathscr{Y}_n = [y_{g1}, y_{g2}, \dots, y_{gn}]$.
* **The Objective:** Given these noisy points, how do we reconstruct the most likely version of the clean signal $X_g$ and, crucially, how much should we trust that reconstruction?

---

**Summary:**

The goal of Gaussian Process Regression is to take **noisy, discrete observations** and produce a **continuous, probabilistic function**.

It doesn't just provide a line; it provides a "corridor of probability" that tells us not only **what** we think the value is, but **how sure we are** based on the proximity of our observations and our prior assumptions about the signal's smoothness.

> **In a sentence:** GPR is the process of using Bayesian inference to "squash" an infinite space of possible functions into a narrow band of likely functions that honor our noisy data.

# Gaussian Process Regression Problem

Let ${G}$ be some set. Consider a one dimensional Gaussian process $X_g\sim \mathscr{N}(\mu_g,\kappa(g,g))$ where $g\in G$ and $\kappa:G\times G \mapsto \mathbb{R}$ is the covariance (also known as the kernel)
$$\mathbb{E}\left((X_g-\mu_g)(X_{g'}-\mu_{g'})\right)=\kappa(g,g').$$

Let $Y_g$ be another stochastic process that satisfies
\begin{align}
Y_g&=X_g+\nu_g,
\end{align}
where $\nu_g\sim \mathscr{N}(\mu,\sigma_m)$.
This represents a noisy sampling of $X_g$. If one samples the stochastic process $Y_g$ at different $g$ and have the set of observations ${y}_{n}\triangleq [y_{g_1},y_{g_2},\cdots,y_{g_n}]^T$, the Gaussian Process (GP) regression is the problem of estimating $X_g$ given the observations $y_n$.


Consider the multidimensional random variable
$\mathscr{Y}_n\triangleq[Y_{g_1},Y_{g_2},\cdots,Y_{g_n}]^T$ and $\mathscr{X}_n\triangleq[X_{g_1},X_{g_2},\cdots,X_{g_n}]^T$. Then $\mathscr{Y}_n\sim  \mathscr{N}(\mu_n,K_n+\sigma_m^2I)$ where, $\mu_n\triangleq[\mu_{g_1}+\mu,\mu_{g_2}+\mu,\cdots,\mu_{g_n}+\mu]^T$ and $K_n$ is the $n\times n$ covariance matrix
\begin{align*}
K_n&\triangleq \mathbb{E}\left((\mathscr{X}_n-\mu_n)(\mathscr{X}_n-\mu_n)^T\right)=
\begin{bmatrix}
\kappa(g_1,g_1) & \kappa(g_1,g_2) & \cdots & \kappa(g_1,g_n)\\
\vdots & \vdots & \vdots &\vdots\\
\kappa(g_n,g_1) & \kappa(g_n,g_2) & \cdots & \kappa(g_n,g_n)
\end{bmatrix}.
\end{align*}


We then have that
\begin{align*}
\left(\begin{array}{c} X_{g}\\ \mathscr{Y}_n
\end{array}\right)\sim\mathscr{N}\left(\begin{bmatrix}  \mu_{g} \\ \mu_n
\end{bmatrix},\begin{bmatrix} \kappa(g,g) & k^T \\ k & K_n+\sigma^2_m
I \end{bmatrix}\right).
\end{align*}

Thus the Gaussian Process (GP) regression problem boils down to finding $\mathbb{E}[X_{g}|\mathscr{Y}_n]$.


# Solution to the Gaussian Process Regression Problem

We have shown in this note on [Multi-Variate-Gaussians](https://github.com/mugalan/introduction-to-statistical-learning/blob/main/Multivariate_Gaussian_Distributions.ipynb) that the conditional random variable satisfies

$$(X_g \mid \mathscr{Y}_n = y_n) \sim \mathscr{N}\Big(\mu_g + k^T(K_n+\sigma_m^2I)^{-1}(y_n - \mu_n), \quad \kappa(g,g) - k^T(K_n+\sigma_m^2I)^{-1}k\Big).$$


Thus the optimal estimate of $X_g$ given the observations $\mathscr{Y}_n=y_n$ is given by:
$$\mathbb{E}[X_g \mid \mathscr{Y}_n=y_n] = \mu_g + k^T(K_n+\sigma_m^2I)^{-1}(y_n - \mu_n),$$
and the uncertainty is given by
$$\mathrm{Var}(X_g \mid \mathscr{Y}_n=y_n)=\kappa(g,g) - k^T(K_n+\sigma_m^2I)^{-1}k.$$

Thus the Gaussian Process Regression problem **boils down to the estimation of an appropriate Kernal (Covariance) $\kappa(g,g')$.**

> While the conditioning identities above provide the mechanism for prediction, the **Kernel** encodes the fundamental physics/logic of the underlying process. Selecting $\kappa(g,g')$ is equivalent to defining the hypothesis space of the regression.

# The Determination of a Suitable Kernel

1. The Kernel: Defining the Function Space
The kernel $\kappa(g, g')$ encodes our prior assumptions about the data. It is a similarity measure; if $\kappa(g, g')$ is large, the model assumes $X_g$ and $X_{g'}$ are highly correlated.
**Common Kernel Choices:**

* **Radial Basis Function (RBF) / Squared Exponential:**
The "gold standard" for smooth processes. It assumes the function is infinitely differentiable.

$$\kappa(g, g') = \sigma_f^2 \exp\left( -\frac{\|g - g'\|^2}{2\ell^2} \right)$$


* **Matérn Kernel:**
More realistic for physical or financial data where "jaggedness" exists. It has a parameter $\nu$ that controls smoothness (e.g., $\nu=1.5$ or $2.5$).
* **Periodic Kernel:**
Used for data with repeating cycles (e.g., daily temperature or seasonal sales).

$$\kappa(g, g') = \sigma_f^2 \exp\left( -\frac{2\sin^2(\pi|g-g'|/p)}{\ell^2} \right)$$



---

2. The Hyperparameter Optimization Problem
    A kernel is rarely "fixed." It usually depends on a vector of hyperparameters $\theta$, such as the **length-scale** ($\ell$), **signal variance** ($\sigma_f^2$), and **noise variance** ($\sigma_m^2$).

    To find the "best" $\theta$, we don't look at the prediction error. Instead, we maximize the **Log-Marginal Likelihood (LML)**. This is a Bayesian way of asking: *"How likely is it that our observed data $\mathbf{y}_n$ was generated by a GP with these specific settings?"*

    **The Objective Function**


    The marginal distribution is given by:
    $$\mathscr{Y}_n \sim \mathscr{N}(\mu_n, K_n + \sigma_m^2 I)$$

    The **Marginal Likelihood** is simply the Probability Density Function (PDF) of this multivariate normal distribution evaluated at your actual observed data points $y_n$.

    $$p(y_n \mid g, \theta) = \frac{1}{\sqrt{(2\pi)^n |K_n + \sigma_m^2 I|}} \exp\left( -\frac{1}{2}(y_n - \mu_n)^T (K_n + \sigma_m^2 I)^{-1} (y_n - \mu_n) \right)$$
    Taking the **natural log** ($\ln$) of this density gives you the
    
    **Log-Marginal Likelihood:**
$$\log p(y_n \mid g, \theta) = \underbrace{-\frac{1}{2}(y_n - \mu_n)^T (K_n + \sigma_m^2 I)^{-1} (y_n - \mu_n)}_{\text{Data Fit}} - \underbrace{\frac{1}{2} \log |K_n + \sigma_m^2 I|}_{\text{Complexity Penalty}} - \underbrace{\frac{n}{2} \log 2\pi}_{\text{Normalization}}$$

**The "Ockham's Razor" Balance**

* **Data Fit:** This term gets better (larger) as the model fits the data points more closely. It encourages "wiggier" kernels that go exactly through the points.
* **Complexity Penalty:** This term gets worse (smaller) as the kernel becomes more complex or the correlation matrix becomes more diverse. It encourages "simpler," smoother kernels.

**The Optimization Task:**
We solve for $\theta^*$ using gradient-based optimizers (like L-BFGS-B):


$$\theta^* = \arg \max_{\theta} \log p(y_n \mid g, \theta)$$

---

**Summary of the Workflow**

1. **Select Kernel Structure:** (e.g., RBF + White Noise).
2. **Initialize $\theta$:** Guess starting values for $\ell$, $\sigma_f^2$, and $\sigma_m^2$.
3. **Optimize:** Maximize the LML to find $\theta^*$. This "tunes" the model to the specific "wiggliness" and "noise" of your dataset.
4. **Predict:** Plug $\theta^*$ into your conditional mean and variance formulas to perform regression.

#Example

The following Python code provides a practical implementation of the **Latent Signal Extraction** problem. Specifically, it demonstrates how GPR can reconstruct a hidden, continuous truth from a small set of "corrupted" data points.

Here is a breakdown of the specific problems the code addresses:

#### 1. The Denoising Problem (Filtering)

In the code, the `y_train` values are generated using a sine wave plus random Gaussian noise.

* **The Code's Solution:** By setting the noise parameter `alpha=0.1`, the model recognizes that the black dots are not the "absolute truth."
* **The Result:** The predictive mean (the green line) provides a smooth, filtered version of the data, effectively separating the **signal** (the underlying physics) from the **noise** (random fluctuations).

#### 2. The Interpolation Problem (Gap Filling)

The training data has significant gaps (e.g., no data exists between $t=3$ and $t=5$).

* **The Code's Solution:** The GPR uses the **Kernel** to calculate the correlation between the known points and the empty spaces.
* **The Result:** It "fills in the blanks" with the most statistically likely path. Notice how the green line naturally follows a curve even where no data points exist.

#### 3. The Uncertainty Quantification Problem

Traditional regression (like a polynomial fit) would give you a line but would be "overconfident"—it wouldn't tell you where it might be wrong.

* **The Code's Solution:** The code calculates the standard deviation $\sigma_g$ for every point along the x-axis.
* **The Result:** The **95% Confidence Interval** (the shaded cloud). You will notice that in the "gaps" between data points, the cloud gets wider. This visually represents the model saying: *"I am making a guess here, but because I haven't seen data in this region, my confidence is lower."*

#### 4. The Hyperparameter "Learning" Problem

The code does not manually set how "wiggly" the curve should be.

* **The Code's Solution:** It uses the **Log-Marginal Likelihood (LML)** optimization we discussed.
* **The Result:** Before fitting, the kernel started with a generic `length_scale=1.0`. After `gp.fit()`, the optimizer looked at the spacing of your data and updated this to an **optimal length-scale**. This ensures the curve is neither too "jittery" nor too "stiff" for the specific data provided.

---

#### Summary for your Documentation

> "The provided Python implementation solves the problem of **Probabilistic Non-Parametric Regression**. It transforms seven noisy, discrete observations into a continuous predictive distribution. The code demonstrates that the GPR is not merely a curve-fitter, but a **risk-aware estimator** that balances empirical data fit with theoretical smoothness assumptions."

In [2]:
import numpy as np
import plotly.graph_objects as go
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C

# 1. Generate Synthetic "Financial" Data
# Imagine this is a normalized stock price over 10 days
np.random.seed(42)
X_train = np.array([1, 3, 5, 6, 7, 8, 9.5]).reshape(-1, 1)
y_train = np.sin(X_train).ravel() + np.random.normal(0, 0.15, X_train.shape[0])

# 2. Define the Kernel (Structure)
# We use an RBF kernel (smoothness) with a constant scaling factor
# Bounds allow the optimizer to "search" for the best hyperparameters
kernel = C(1.0, (1e-3, 1e3)) * RBF(1.0, (1e-2, 1e2))

# 3. GP Regression & Hyperparameter Optimization
# n_restarts_optimizer handles the non-convex LML surface
gp = GaussianProcessRegressor(kernel=kernel, alpha=0.1, n_restarts_optimizer=10)
gp.fit(X_train, y_train)

# 4. Predict over a dense range for plotting
X_plot = np.linspace(0, 11, 200).reshape(-1, 1)
y_mean, y_std = gp.predict(X_plot, return_std=True)

# 5. Visualize with Plotly
fig = go.Figure()

# Add 95% Confidence Interval (The Uncertainty Cloud)
fig.add_trace(go.Scatter(
    x=np.concatenate([X_plot.ravel(), X_plot.ravel()[::-1]]),
    y=np.concatenate([y_mean - 1.96 * y_std, (y_mean + 1.96 * y_std)[::-1]]),
    fill='toself',
    fillcolor='rgba(0,100,80,0.2)',
    line_color='rgba(255,255,255,0)',
    name='95% Confidence Interval',
    showlegend=True
))

# Add the Predictive Mean (The Estimate)
fig.add_trace(go.Scatter(
    x=X_plot.ravel(), y=y_mean,
    line=dict(color='rgb(0,100,80)'),
    name='Predictive Mean (X_g)'
))

# Add the actual observations (The Noisy Data)
fig.add_trace(go.Scatter(
    x=X_train.ravel(), y=y_train,
    mode='markers',
    marker=dict(color='black', size=10),
    name='Observations (Y_n)'
))

# Formatting
fig.update_layout(
    title=f"GPR Result (Optimized Length-scale: {gp.kernel_.get_params()['k2__length_scale']:.2f})",
    xaxis_title="Time (g)",
    yaxis_title="Price / Value (X_g)",
    template="plotly_white",
    hovermode="x"
)

fig.show()

# Linear regression

To show how **Linear Regression** is a specific case of Gaussian Process Regression, we simply need to choose a specific **Kernel** $\kappa(g, g')$ that represents a linear relationship.

In linear regression, we assume the underlying function $X_g$ takes the form:


$$X_g = g^T w$$


where $w$ is a vector of weights with a Gaussian prior $w \sim \mathscr{N}(0, \Sigma_w)$.

---

### 1. Defining the Linear Kernel

If we assume $X_g = g^T w$, the covariance (kernel) between any two points $g$ and $g'$ is derived as follows:
\begin{align*}
\kappa(g, g') &= \mathbb{E}[X_g X_{g'}] \
&= \mathbb{E}[(g^T w)(g'^T w)^T] \
&= g^T \mathbb{E}[ww^T] g' \
&= g^T \Sigma_w g'
\end{align*}

If we assume the weights are independent and have variance $\sigma_w^2$ (i.e., $\Sigma_w = \sigma_w^2 I$), the kernel becomes:


$$\kappa(g, g') = \sigma_w^2 g^T g'$$


This is known as the **Linear Kernel**.

---

### 2. Applying the GPR Framework

Using your established notation, let’s substitute this linear kernel into the $K_n$ and $k$ terms. Let $G$ be the matrix of training inputs $[g_1, g_2, \dots, g_n]^T$. Then:

* $K_n = \sigma_w^2 G G^T$
* $k^T = \sigma_w^2 g^T G^T$

Now, substitute these into your **Optimal Estimate** formula (assuming $\mu_g = 0$ and $\mu_n = 0$ for simplicity):


$$\mathbb{E}[X_g \mid \mathscr{Y}_n=y_n] = \underbrace{\sigma_w^2 g^T G^T}_{k^T} (\underbrace{\sigma_w^2 G G^T + \sigma_m^2 I}_{K_n + \sigma_m^2 I})^{-1} y_n$$

---

### 3. The Matrix Identity (The "Bridge")

To see the familiar linear regression form, we use the **Matrix Inversion Lemma** (Woodbury Identity):


$$\sigma_w^2 G^T (\sigma_w^2 G G^T + \sigma_m^2 I)^{-1} = (G^T G + \frac{\sigma_m^2}{\sigma_w^2} I)^{-1} G^T$$

Substituting this back into the expectation:


$$\mathbb{E}[X_g \mid \mathscr{Y}_n=y_n] = g^T \underbrace{\left( G^T G + \frac{\sigma_m^2}{\sigma_w^2} I \right)^{-1} G^T y_n}_{\hat{w}}$$

---

### 4. Conclusion: Ridge Regression

The term we labeled $\hat{w}$ is exactly the solution for **Ridge Regression** (Linear Regression with $L_2$ regularization):


$$\hat{w} = (G^T G + \lambda I)^{-1} G^T y_n$$


where the regularization parameter $\lambda = \frac{\sigma_m^2}{\sigma_w^2}$ is the ratio of **observation noise** to **prior weight variance**.

### Summary of the Relationship

| Feature | Gaussian Process Regression | Linear Regression |
| --- | --- | --- |
| **Kernel** | General $\kappa(g, g')$ | Linear $\sigma_w^2 g^T g'$ |
| **Function Space** | Infinite-dimensional (flexible) | Finite-dimensional (straight lines) |
| **Regularization** | Handled by noise $\sigma_m^2$ | Handled by $\lambda = \sigma_m^2/\sigma_w^2$ |

> **The Insight:** Linear regression is simply a Gaussian Process restricted by a kernel that only allows "flat" functions. By changing the kernel from linear to RBF, you effectively move from "Linear Regression" to "Infinite-Dimensional Non-Linear Regression."

Would you like to see how a **Polynomial Kernel** can similarly bridge the gap to Polynomial Regression?

###GP - Approach

Consider the stochastic process defined above and let $G=\mathbb{R}^n$,  $\nu\sim \mathscr{N}(0,\sigma_m^2I)$, and $X_g=g^Tw$ for $w\sim\mathscr{N}(\mu_w,\sigma_w^2I)$. Then $\kappa(g,g')=g^T\Sigma_wg'$ and the above expression becomes
\begin{align}
Y_g&=X_g+\nu=g^Tw+\nu.
\end{align}
The GP regression problem reduces in this case to that of the linear regression problem and the above expressions reduce to
\begin{align}
\widehat{\mu}_{g}&=\mu_w+k^T (K_n+\sigma^2_mI)^{-1}(y_n-\mu_n),\\
\sigma^2_{g}&= \sigma^2_m+\kappa(g,g)-k^T (K_n+\sigma^2_mI)^{-1}k.
\end{align}


### Bayesian - Approach

Consider a set of data $\{y_i=y(g_i)\}$ where $G=\mathbb{R}^n$. We will attmpt to model the data by a random process
\begin{align}
Y_g&=g^Tw,
\end{align}
where $X_g=g^Tw$ for $w$ some random variable.
\begin{align}
\mathscr{P}(w\,|\,\{Y_g=y_g\})&\propto\mathscr{P}(\{Y_g=y_g\}\,|\,w)\mathscr{P}(w).
\end{align}

If we assume $w\sim\mathscr{N}(\mu_w,\Sigma_w)$ then from properties of conditional normal distributions
\begin{align*}
p\left(\begin{array}{c}  w \\ \mathscr{Y}_n
\end{array}\right)=\mathscr{N}\left(\begin{bmatrix}  \mu_w \\ \mu_n
\end{bmatrix},\begin{bmatrix} \kappa(g,g) & k^T \\ k & K_n \end{bmatrix}\right).
\end{align*}

\begin{align}
\mathbb{E}\left(\begin{bmatrix} w-\mu_w\\ \mathscr{Y}_n-\mu_n\end{bmatrix}\begin{bmatrix}(w-\mu_w)^T & (\mathscr{Y}_n-\mu_n)^T\end{bmatrix}\right)
\end{align}

Let $\mathscr{P}(w\mid \{\mathscr{Y}_n=y_n\})\sim \mathscr{N}(\widehat{\mu}_{w},\widehat{\Sigma}_{w})$. From the properties of normal distributions (please refer to \cite{} for formulas for constructing the conditional probabilities of Gaussian distributions) we find that

\begin{align}
\widehat{\mu}_{w}&=\mu_w+k^T K_n^{-1}(y_n-\mu_n),\\
\widehat{\Sigma}&= \Sigma_w-k^T K_n^{-1}k.
\end{align}
where
\begin{align}
k&=E\left((\mathscr{Y}_n-\mu_n)(w-\mu_{w})^T\right)=\begin{bmatrix}g_1^T\\\vdots\\ g_n^T\end{bmatrix}E\left((w-\mu_{w})(w-\mu_{w})^T\right)=\begin{bmatrix}g_1^T\\\vdots\\ g_n^T\end{bmatrix}\Sigma_w.
\end{align}
and
\begin{align}
({K_n})_{ij}=E\left((Y_{g_i}-\mu_{g_i})(Y_{g_j}-\mu_{g_j})\right)=g_i^T\Sigma_wg_j
\end{align}